# CAME Assignment 02: inference and model comparison

This notebook does **not** train anything. It takes one or more upload-ready submission files
(`assignment02.py` with `TRAINED_MODEL_B64` filled in), loads each one exactly the way CMS does, and compares them:

1. `load_trained_model(cpu)` → `predict()` → `classification_report()`: the module's own functions, the same path CMS Stage B uses.
2. **Validation set**: the same 2,700 patches `main()` used (2,000 per class, 85/15 split, seed 42). Training used this set to pick the best epoch, so the numbers are a little optimistic.
3. **External test set** (optional): `CRC_VAL_HE_7K`, about 7,180 patches from different patients that neither run ever saw. This is the fairest way to compare two runs.
4. Confusion matrices, per-class recall, misclassified patches, and single-image inference with top-3 probabilities.
5. Downloads the best candidate as `assignment02.py`.

Runtime: CPU is enough (CMS grades on CPU). A GPU only makes section 5 faster.

## 0. Install

In [ ]:
# torch / torchvision / matplotlib / pandas are already on Colab; these two are for the Hugging Face data.
!pip -q install datasets huggingface_hub

## 1. Google Drive

The working folder is the one the training notebook used, so the cached subset `nct_crc_he_100k_subset_2000.pt` loads without downloading again.

In [ ]:
import os
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
WORK_DIR = Path('/content/drive/MyDrive/came_a02')
CAND_DIR = WORK_DIR / 'candidates'          # submission files to compare live here
CAND_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
print('Working directory:', os.getcwd())
print('Cached subsets   :', sorted(p.name for p in WORK_DIR.glob('*.pt')))
print('Candidates folder:', sorted(p.name for p in CAND_DIR.glob('*.py')))

## 2. Candidate submission files

Put the files to compare in `came_a02/candidates/` on Drive, or let the cell below open an upload dialog.
From the local `work` folder:

| key | file in `candidates/` | local file |
|---|---|---|
| `new` | `candidate_new.py` | `assignment02_submission.py` (latest run) |
| `old_best` | `candidate_old.py` | `upload/assignment02.py` (earlier best run) |

If a file is missing, upload it from the dialog. Uploaded files keep their own names, and each file's name without `.py` becomes its key.
Uploads go to `candidates/`, so they never overwrite the training `assignment02.py`.

Expected fingerprints (sha256 prefix of the base64 string):
`new` = `4733C7E2EF42E22D`, `old_best` = `9D0ED8B25A2FF6E4`. Code (without weights) = `29FD9E42A462B8EF` for both.

In [ ]:
import hashlib
import importlib.util
import re
import sys

# key -> file name inside CAND_DIR. Add or remove entries freely.
CANDIDATES = {
    'new': 'candidate_new.py',        # local assignment02_submission.py (latest run)
    'old_best': 'candidate_old.py',   # local upload/assignment02.py (earlier best run)
}

missing = [f for f in CANDIDATES.values() if not (CAND_DIR / f).exists()]
if missing:
    from google.colab import files
    print('Not found in', CAND_DIR, ':', missing)
    print('Select the submission .py files to compare (several at once is fine).')
    os.chdir(CAND_DIR)                 # files.upload() saves into the current directory
    try:
        uploaded = files.upload()
    finally:
        os.chdir(WORK_DIR)
    CANDIDATES = {k: v for k, v in CANDIDATES.items() if (CAND_DIR / v).exists()}
    for fname in uploaded:
        if fname.endswith('.py') and fname not in CANDIDATES.values():
            CANDIDATES[Path(fname).stem] = fname
assert CANDIDATES, 'No candidate files to evaluate.'


def b64_fingerprint(b64: str) -> str:
    return hashlib.sha256(re.sub(r'\s', '', b64).encode('ascii')).hexdigest()[:16].upper()


def code_fingerprint(path: Path) -> str:
    text = path.read_text(encoding='utf-8').replace('\r\n', '\n')
    code = re.sub(r'(?ms)^TRAINED_MODEL_B64 = """.*?"""', 'X', text)
    return hashlib.sha256(code.encode('utf-8')).hexdigest()[:16].upper()


def load_candidate(key: str, path: Path):
    """Import a submission file as its own module (import only: no training, no download)."""
    mod_name = 'cand_' + re.sub(r'\W', '_', key)
    spec = importlib.util.spec_from_file_location(mod_name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = mod
    spec.loader.exec_module(mod)
    assert mod.TRAINED_MODEL_B64.strip(), f'{path.name}: TRAINED_MODEL_B64 is EMPTY (0-mark gate on CMS)'
    return mod


modules = {}
for key, fname in CANDIDATES.items():
    path = CAND_DIR / fname
    modules[key] = load_candidate(key, path)
    print(f'{key:10s} {fname:30s} weights={b64_fingerprint(modules[key].TRAINED_MODEL_B64)} '
          f'code={code_fingerprint(path)} size={path.stat().st_size:,} B')

## 3. Validation set (same split as training)

Uses the first candidate's own `build_datasets()` (the data code is the same in every candidate) with the training settings:
`CAME_PER_CLASS=2000` and Hugging Face streaming, no `CAME_DATA_DIR`. With the cache file in `came_a02/` this takes a few seconds. Without it, the patches are streamed again in the same fixed order, which gives the same subset.

In [ ]:
import torch
from torch.utils.data import DataLoader

os.environ.pop('CAME_DATA_DIR', None)   # Hugging Face path, not the CMS ImageFolder path
os.environ['CAME_PER_CLASS'] = '2000'   # same subset as the training run
os.environ['CAME_HF_MODE'] = 'stream'

ref = next(iter(modules.values()))
CLASSES = ref.official_classes()
_, val_dataset = ref.build_datasets()
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2)
y_val = val_dataset.labels.clone().long()
print('validation patches:', len(val_dataset))
print('per class         :', dict(zip(CLASSES, torch.bincount(y_val, minlength=9).tolist())))

## 4. External test set: CRC-VAL-HE-7K (optional)

This set comes from different patients than NCT-CRC-HE-100K and was never used for training or for choosing a checkpoint, so it is the fairest way to compare runs.
Expect scores a few points below validation; that drop is normal for this dataset.
On the first run it streams the split once (patches resized to 64×64 like training) and caches it as `crc_val_he_7k_64px.pt` in `came_a02/`.
Set `RUN_EXTERNAL_TEST = False` to skip it.

In [ ]:
RUN_EXTERNAL_TEST = True
EXT_SPLIT = 'CRC_VAL_HE_7K'
EXT_CACHE = WORK_DIR / 'crc_val_he_7k_64px.pt'


def load_external_test():
    if EXT_CACHE.exists():
        print('[ext] loading cache', EXT_CACHE.name)
        blob = torch.load(EXT_CACHE, map_location='cpu', weights_only=True)
        return blob['images'], blob['labels']

    from datasets import Image as HFImage
    from datasets import load_dataset
    from huggingface_hub import HfFileSystem

    fs = HfFileSystem()
    repo = f'datasets/{ref.HF_DATASET_ID}/data'
    found = sorted(fs.glob(f'{repo}/{EXT_SPLIT}-*.parquet'))
    if not found:
        splits = sorted({f.rsplit('/', 1)[-1].split('-')[0] for f in fs.glob(f'{repo}/*.parquet')})
        raise FileNotFoundError(f'{EXT_SPLIT} not in {ref.HF_DATASET_ID}; available splits: {splits}')
    shards = ['data/' + f.rsplit('/', 1)[-1] for f in found]
    print(f'[ext] streaming {EXT_SPLIT}: {len(shards)} shard(s)')

    ds = load_dataset(ref.HF_DATASET_ID, data_files={EXT_SPLIT: shards}, split=EXT_SPLIT, streaming=True)
    ds = ds.cast_column('image', HFImage(decode=False))
    to_idx = ref._label_mapper(ds.features)
    images, labels = [], []
    for i, example in enumerate(ds):
        y = to_idx(example['label'])
        if 0 <= y < ref.NUM_CLASSES:
            images.append(ref._patch_to_uint8(example['image']))
            labels.append(y)
        if (i + 1) % 1000 == 0:
            print(f'[ext] read {i + 1} patches')
    images = torch.stack(images).to(torch.uint8)
    labels = torch.tensor(labels, dtype=torch.long)
    torch.save({'images': images, 'labels': labels}, EXT_CACHE)
    print('[ext] cached to', EXT_CACHE.name)
    return images, labels


ext_dataset, ext_loader, y_ext = None, None, None
if RUN_EXTERNAL_TEST:
    try:
        ext_images, y_ext = load_external_test()
        ext_dataset = ref.NCTPatchDataset(ext_images, y_ext, ref.make_transforms(False))
        ext_loader = DataLoader(ext_dataset, batch_size=256, shuffle=False, num_workers=2)
        print('external test patches:', len(ext_dataset))
        print('per class            :', dict(zip(CLASSES, torch.bincount(y_ext, minlength=9).tolist())))
    except Exception as exc:
        ext_dataset, ext_loader, y_ext = None, None, None
        print('External test set skipped:', exc)

## 5. Inference with every candidate

Each candidate runs its own `load_trained_model()`, `predict()` and `classification_report()`, the functions CMS calls.
`DEVICE` is the CPU because CMS grades on CPU. `'cuda'` is faster and gives the same predictions.
PDF Stage B: macro recall ≥ 0.80 → 35/35.

In [ ]:
import time

import numpy as np
import pandas as pd

DEVICE = torch.device('cpu')
PASS_MARK = 0.80


def evaluate(mod, loader, y_true):
    model = mod.load_trained_model(DEVICE)
    t0 = time.time()
    y_pred = mod.predict(model, loader, DEVICE)
    seconds = time.time() - t0
    rep = mod.classification_report(y_true, y_pred)
    conf = rep['confusion'].double()
    support = conf.sum(dim=1)
    per_class = [float(conf[i, i] / support[i]) if support[i] > 0 else float('nan') for i in range(9)]
    return {'acc': rep['accuracy'], 'macro_recall': rep['macro_recall'], 'per_class': per_class,
            'confusion': rep['confusion'], 'y_pred': y_pred, 'seconds': seconds}


SETS = {'val': (val_loader, y_val)}
if ext_loader is not None:
    SETS['ext'] = (ext_loader, y_ext)

results = {}
for key, mod in modules.items():
    results[key] = {}
    for split, (loader, y_true) in SETS.items():
        r = evaluate(mod, loader, y_true)
        results[key][split] = r
        print(f'{key:10s} {split:4s} acc={r["acc"]:.4f} macro_recall={r["macro_recall"]:.4f} '
              f'min_class={np.nanmin(r["per_class"]):.4f} ({r["seconds"]:.1f}s on {DEVICE})')

rows = []
for key, res in results.items():
    row = {'candidate': key}
    for split, r in res.items():
        row[f'{split}_acc'] = r['acc']
        row[f'{split}_macro_recall'] = r['macro_recall']
        row[f'{split}_min_class_recall'] = np.nanmin(r['per_class'])
    rows.append(row)
summary = pd.DataFrame(rows).set_index('candidate').round(4)
display(summary)

per_class = pd.DataFrame(
    {f'{key} | {split}': r['per_class'] for key, res in results.items() for split, r in res.items()},
    index=CLASSES).round(4)
display(per_class)

# Rank by the unseen external set when it is available; validation picked the checkpoints, so it is optimistic.
RANK_BY = 'ext_macro_recall' if 'ext_macro_recall' in summary.columns else 'val_macro_recall'
best = summary[RANK_BY].idxmax()
print(f'\nRanking by {RANK_BY}:')
print(summary[RANK_BY].sort_values(ascending=False).to_string())
print(f'\nBest candidate: {best}')
gap = summary[RANK_BY].max() - summary[RANK_BY].min()
if len(summary) > 1 and gap < 0.003:
    print(f'Note: the gap is only {gap:.4f}, which is within noise. Either file is fine.')
for key in summary.index:
    ok = summary.loc[key, 'val_macro_recall'] >= PASS_MARK
    print(f'{key:10s} Stage B (val macro recall >= {PASS_MARK}): {"PASS" if ok else "BELOW"}')

## 6. Confusion matrices and per-class recall

Each row is normalized, so the diagonal is that class's recall.

In [ ]:
import matplotlib.pyplot as plt


def plot_confusions(split):
    keys = [k for k in results if split in results[k]]
    fig, axes = plt.subplots(1, len(keys), figsize=(5.4 * len(keys), 4.8), squeeze=False)
    for ax, key in zip(axes[0], keys):
        cm = results[key][split]['confusion'].double()
        cm = (cm / cm.sum(dim=1, keepdim=True).clamp(min=1)).numpy()
        ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
        ax.set_xticks(range(9))
        ax.set_xticklabels(CLASSES, rotation=45)
        ax.set_yticks(range(9))
        ax.set_yticklabels(CLASSES)
        for i in range(9):
            for j in range(9):
                if cm[i, j] >= 0.01:
                    ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center', fontsize=7,
                            color='white' if cm[i, j] > 0.5 else 'black')
        ax.set_title(f'{key} | {split} | macro recall {results[key][split]["macro_recall"]:.4f}', fontsize=10)
        ax.set_xlabel('predicted')
        ax.set_ylabel('true')
    plt.tight_layout()
    plt.show()


def plot_per_class(split):
    keys = [k for k in results if split in results[k]]
    x = np.arange(9)
    w = 0.8 / len(keys)
    plt.figure(figsize=(11, 3.8))
    for n, key in enumerate(keys):
        plt.bar(x - 0.4 + w * (n + 0.5), results[key][split]['per_class'], w, label=key)
    plt.axhline(PASS_MARK, color='red', ls='--', lw=1, label=f'{PASS_MARK:.2f}')
    plt.xticks(x, CLASSES)
    plt.ylim(0, 1.02)
    plt.ylabel('recall')
    plt.title(f'Per-class recall ({split})')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()


for split in SETS:
    plot_confusions(split)
    plot_per_class(split)

## 7. Misclassified patches

A random sample of the patches the chosen candidate gets wrong. T = true class, P = predicted class.

In [ ]:
CHOSEN = best   # or pick by key, e.g. 'old_best'


def show_mistakes(key, split='val', n=24):
    dataset, y_true = (val_dataset, y_val) if split == 'val' else (ext_dataset, y_ext)
    if dataset is None:
        print(f'No {split} data loaded.')
        return
    y_pred = results[key][split]['y_pred']
    wrong = (y_pred != y_true).nonzero(as_tuple=False).view(-1)
    print(f'{key} | {split}: {wrong.numel()} wrong out of {y_true.numel()}')
    if wrong.numel() == 0:
        return
    pick = wrong[torch.randperm(wrong.numel(), generator=torch.Generator().manual_seed(0))[:n]].tolist()
    cols = 8
    rows = (len(pick) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.95), squeeze=False)
    for ax in axes.reshape(-1):
        ax.axis('off')
    for ax, i in zip(axes.reshape(-1), pick):
        ax.imshow(dataset.images[i].numpy())
        ax.set_title(f'T:{CLASSES[int(y_true[i])]}\nP:{CLASSES[int(y_pred[i])]}', fontsize=8, color='red')
    plt.tight_layout()
    plt.show()


for split in SETS:
    show_mistakes(CHOSEN, split)

## 8. Single-image inference (top-3 probabilities)

`classify()` takes a PIL image or a file path and returns the top-3 classes with their softmax probabilities. It applies the same preprocessing as validation: `make_transforms(False)`, which resizes to 64×64 and normalizes.
The demo shows one validation patch per class (the validation set is ordered by class, 300 per class).

In [ ]:
import io

import torch.nn.functional as F
from PIL import Image

infer_mod = modules[CHOSEN]
infer_model = infer_mod.load_trained_model(DEVICE)
infer_tf = infer_mod.make_transforms(False)


def classify(img, top_k=3):
    if not isinstance(img, Image.Image):
        img = Image.open(img)
    x = infer_tf(img.convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = F.softmax(infer_model(x), dim=1)[0].cpu()
    top = prob.topk(top_k)
    return [(CLASSES[int(i)], float(p)) for p, i in zip(top.values, top.indices)]


per_class_n = len(val_dataset) // 9
demo_idx = [c * per_class_n for c in range(9)]
fig, axes = plt.subplots(1, 9, figsize=(18, 2.6))
for ax, i in zip(axes, demo_idx):
    img = Image.fromarray(val_dataset.images[i].numpy())
    top = classify(img)
    ax.imshow(img)
    ax.axis('off')
    ok = top[0][0] == CLASSES[int(y_val[i])]
    ax.set_title(f'true {CLASSES[int(y_val[i])]}\n{top[0][0]} {top[0][1]:.2f}', fontsize=9,
                 color='green' if ok else 'red')
    print(f'true={CLASSES[int(y_val[i])]:5s} -> ' + ', '.join(f'{c} {p:.3f}' for c, p in top))
plt.tight_layout()
plt.show()

## 9. Classify your own images (optional)

Upload H&E patches (`.tif`, `.png` or `.jpg`). Each one is resized to 64×64 like the training data, so 224×224 NCT patches work as they are.

In [ ]:
from google.colab import files

uploaded_imgs = files.upload()
for fname, data in uploaded_imgs.items():
    img = Image.open(io.BytesIO(data)).convert('RGB')
    top = classify(img)
    plt.figure(figsize=(2.4, 2.6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'{top[0][0]} ({top[0][1]:.2f})', fontsize=10)
    plt.show()
    print(f'{fname}: ' + ', '.join(f'{c} {p:.3f}' for c, p in top))

## 10. Download the chosen file as `assignment02.py`

This copies the `CHOSEN` candidate (section 7) unchanged and renames it to `assignment02.py`, the name CMS expects.

In [ ]:
import shutil

from google.colab import files

out_dir = Path('/content/best_upload')
out_dir.mkdir(exist_ok=True)
dst = out_dir / 'assignment02.py'
shutil.copyfile(CAND_DIR / CANDIDATES[CHOSEN], dst)
print(f'{CHOSEN}: weights={b64_fingerprint(modules[CHOSEN].TRAINED_MODEL_B64)} '
      f'size={dst.stat().st_size:,} B -> {dst}')
files.download(str(dst))